# GenreHalluEval, single best judge (A)

The single best judge on BenHalluEval (**A = gemma4-31b**, in both conditions) judges every
answer. No genre is needed.

**Conditions:** each answer is judged once, in its own condition (`condition` column of `items.csv`, Set B):
`context` = passage + question + answer (`judge_v1`); `no_context` = question + answer (`judge_noctx_v1`).
The prompts, settings, Ollama version and parsing are exactly those of every BenHalluEval run; the notebook stops if
they differ. This notebook only collects verdicts; `src/dataset2/compare_routes.py` scores and compares the two
notebooks afterwards.

## How to run it on Kaggle

1. **Settings:** GPU T4 x2, Internet on.
2. **Add Input:** your private Kaggle dataset **GenreHalluEval** (made from `GenreHalluEval.zip`).
3. Run all. Keep `RUN_MODE = "full"` (a `"pilot"` of 20 questions is only for a quick check).
4. When it ends, download `genrehallueval__single_best__full__outputs.zip` (Output tab) and give it to Claude.
5. **If the run stops** (time limit, error): "Save Version", then in a new version **Add Input → this notebook's
   output**, and run all again. Finished calls are skipped. Add only this notebook's own earlier output.

In [ ]:
# ======================= EDIT ONLY THIS CELL =======================
RUN_MODE = "full"           # "full" = all 250 questions; "pilot" = 20 questions (4 per genre), a quick check only
TIME_BUDGET_HOURS = 11.0    # stop cleanly before Kaggle's 12-hour limit; then resume (step 5 above)
OLLAMA_VERSION = "0.40.6"   # the Ollama of every BenHalluEval run. Do not change
# ===================================================================
NOTEBOOK = "single_best"
METHODS = ["A"]
MODEL_KEY = "gemma4-31b"    # the starting model; the notebook switches models by itself
PARALLEL = None             # requests at once: each model's default
STAGES = ["judge"]
JUDGE_CONDITIONS = ["context", "no_context"]   # each answer is judged once, in its own condition

In [ ]:
import collections, csv, datetime, glob, hashlib, json, math, os, platform, random, re, shutil
import statistics, subprocess, sys, threading, time, urllib.error, urllib.request, zipfile
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

T0 = time.time()
SEED = 2026
DATASET = "genrehallueval"          # Dataset 2, test-only: this notebook never makes or changes a choice
GENRES = {"HISTORY": "History, Politics & Education", "ARTS": "Arts, Culture & Sports",
          "SCIENCE": "Science & Technology", "GEOGRAPHY": "Geography & Places",
          "RELIGION": "Religion", "OTHER": "Other"}
CODE_OF = {name: code for code, name in GENRES.items()}
MIN_CELL = 50                       # golden rule 5
PILOT_QUESTIONS = 40
BOOT = 1000


class Stop(Exception):
    """A clear, deliberate stop with an instruction for the user."""


try:
    from IPython.display import Markdown, display

    def show_md(text):
        display(Markdown(text))
except Exception:
    def show_md(text):
        print(text)


def find_data_dir():
    if os.environ.get("BENCH_DATA_DIR"):
        return Path(os.environ["BENCH_DATA_DIR"])
    hits = []
    for depth in ("*", "*/*", "*/*/*", "*/*/*/*"):
        hits += glob.glob(f"/kaggle/input/{depth}/items.csv")
    hits = sorted({str(Path(h).parent) for h in hits if (Path(h).parent / "lock_v2.json").exists()})
    if len(hits) != 1:
        raise Stop(f"Expected exactly one GenreHalluEval dataset (a folder with items.csv and lock_v2.json) under "
                   f"/kaggle/input, found {hits}. Add it with 'Add Input' (step 2 at the top).")
    return Path(hits[0])


DATA_DIR = find_data_dir()
OUT_DIR = Path(os.environ.get("BENCH_OUT_DIR", "/kaggle/working"))
RUN_DIR, RES_DIR = OUT_DIR / "runs", OUT_DIR / "results"
RUN_DIR.mkdir(parents=True, exist_ok=True)
RES_DIR.mkdir(parents=True, exist_ok=True)
assert RUN_MODE in ("pilot", "full"), RUN_MODE
assert STAGES and set(STAGES) <= {"genre", "judge"}, STAGES
assert JUDGE_CONDITIONS and set(JUDGE_CONDITIONS) <= {"context", "no_context"}, JUDGE_CONDITIONS
COND_NAME = {"context": "WITH the passage", "no_context": "WITHOUT the passage"}
print("dataset:", DATA_DIR)
print("outputs:", OUT_DIR)

## The judge models (the same table as the BenHalluEval notebook)

In [ ]:
DEFAULTS = dict(think=None, max_new={"genre": 8, "judge": 8}, parallel=4, max_ctx=32768, expect_arch=None,
                revision="main", note="")
MODELS = {
    # ---- the paper's 9 judges ----
    "llama3.1-8b": dict(backend="ollama", model="llama3.1:8b", pool="open", source="paper"),
    "mistral-nemo-12b": dict(backend="ollama", model="mistral-nemo:12b", pool="open", source="paper"),
    "gemma2-27b": dict(backend="ollama", model="gemma2:27b", pool="open", source="paper", parallel=2,
                       max_ctx=8192),
    "qwen2.5-32b": dict(backend="ollama", model="qwen2.5:32b", pool="open", source="paper", parallel=2),
    "deepseek-r1-14b": dict(backend="ollama", model="deepseek-r1:14b", pool="open", source="paper", think=True,
                            max_new={"genre": 1024, "judge": 1024},
                            note="thinking capped at 1,024 tokens; capped replies count as no_answer"),
    "tigerllm-9b": dict(backend="ollama", model="hf.co/mradermacher/TigerLLM-9B-it-GGUF:Q4_K_M", pool="open",
                        source="paper", expect_arch="gemma3", max_ctx=8192,
                        note="GGUF (2025-09-15) of the Sept 2025 TigerLLM-9B-it upload: Gemma-3, ~12B despite the name"),
    "titullm-3b": dict(backend="ollama", model="hf.co/mradermacher/titulm-llama-3.2-3b-v1.1-GGUF:Q4_K_M",
                       pool="open", source="paper", expect_arch="llama",
                       note="hishab/titulm-llama-3.2-3b-v1.1; check it is the checkpoint the paper used"),
    "banglallama-13b": dict(backend="hf", model="BanglaLLM/bangla-llama-13b-instruct-v0.1", pool="open",
                            source="paper", max_ctx=4096, lora_merge=dict(r=64, alpha=128), batch_tokens=6144,
                            note="LLaMA-2 based; the checkpoint stores un-merged LoRA layers, merged here with "
                                 "r=64, alpha=128 (its training script); prompts over 4,096 tokens are skipped"),
    "titullm-3b-instruct": dict(backend="hf", model="hishab/titulm-llama-3.2-3b-v2.0-Instruct-v1.0",
                                revision="6a9600483daf9a3af7494e7fa91f2e72e2799b96", pool="open",
                                source="new (instruct TituLLM, added 2026-10-18)", max_ctx=4096, parallel=8,
                                batch_tokens=16384,
                                note="instruction-tuned TituLLM (not the paper's base checkpoint); own chat template"),
    "gpt-4.1-mini": dict(backend="openai", model="gpt-4.1-mini-2025-04-14", pool="closed", source="paper",
                         parallel=8, max_ctx=1_000_000, note="secondary pool only; paid API"),
    # ---- new open models (2025-26) ----
    "gemma4-31b": dict(backend="ollama", model="gemma4:31b", pool="open", source="new", parallel=2),
    "qwen3.5-27b": dict(backend="ollama", model="qwen3.5:27b", pool="open", source="new", think=False,
                        parallel=2),
    "gpt-oss-20b": dict(backend="ollama", model="gpt-oss:20b", pool="open", source="new (optional)",
                        think="low", max_new={"genre": 512, "judge": 512}),
}
PAPER_QA_BHS = {"deepseek-r1-14b": 38.60, "gpt-4.1-mini": 15.56, "qwen2.5-32b": 46.85, "gemma2-27b": 47.45,
                "mistral-nemo-12b": 53.84, "llama3.1-8b": 47.71, "tigerllm-9b": 28.28, "titullm-3b": 50.00,
                "banglallama-13b": 48.62}   # zero-shot QA, paper Table 4 (other answers, so only a rough guide)
PRICE_PER_M = {"gpt-4.1-mini-2025-04-14": (0.40, 1.60)}  # USD per 1M input/output tokens; check current prices

if MODEL_KEY not in MODELS:
    raise Stop(f"MODEL_KEY must be one of: {', '.join(MODELS)}")
CFG = {**DEFAULTS, **MODELS[MODEL_KEY]}
if "claude" in CFG["model"].lower():
    raise Stop("Claude wrote the genre drafts and the hallucinated answers: never use it as a judge or guesser.")
if os.environ.get("BENCH_BACKEND_OVERRIDE"):          # local tests only
    CFG["backend"] = os.environ["BENCH_BACKEND_OVERRIDE"]
PAR = PARALLEL or CFG["parallel"]
show_md("| key | backend | model | pool | source | parallel | note |\n|---|---|---|---|---|---|---|\n" + "\n".join(
    f"| {'**' + k + '**' if k == MODEL_KEY else k} | {m['backend']} | `{m['model']}` | {m['pool']} | {m['source']} "
    f"| {m.get('parallel', DEFAULTS['parallel'])} | {m.get('note', '')} |" for k, m in MODELS.items()))
print(f"\nThis run: {MODEL_KEY} | mode {RUN_MODE} | stages {STAGES} | conditions {JUDGE_CONDITIONS} | parallel {PAR}")

## Load GenreHalluEval and the lock

The files must match `checksums.sha256`, and the lock must be exactly `lock_v2.json` (sha256 `6d3c1d29…`).

In [ ]:
LOCK_SHA256 = "6d3c1d29cb8159d5f74300c799ce040c9a7c737dfb68b5c292a2a2aa948c9137"
PILOT_PER_GENRE = 4


def read_csv(path):
    with open(path, encoding="utf-8-sig", newline="") as f:
        return list(csv.DictReader(f))


sums = {}
for line in (DATA_DIR / "checksums.sha256").read_text(encoding="utf-8").splitlines():
    if line.strip():
        digest, name = line.split("  ", 1)
        sums[name.strip()] = digest
for name in ["items.csv", "lock_v2.json", "prompts/genre_v1.txt", "prompts/judge_v1.txt", "prompts/judge_noctx_v1.txt"]:
    if hashlib.sha256((DATA_DIR / name).read_bytes()).hexdigest() != sums.get(name):
        raise Stop(f"{name} does not match checksums.sha256. Re-upload the dataset folder unchanged.")
if hashlib.sha256((DATA_DIR / "lock_v2.json").read_bytes()).hexdigest() != LOCK_SHA256:
    raise Stop("lock_v2.json is not the locked file. Never change the lock.")
LOCK = json.loads((DATA_DIR / "lock_v2.json").read_text(encoding="utf-8"))
POOL = LOCK["pools"]["dataset2_pool"]
missing = [k for k in POOL if k not in MODELS]
if missing:
    raise Stop(f"pool judges missing from the model table: {missing}")

ITEMS = read_csv(DATA_DIR / "items.csv")
if len({r["item_id"] for r in ITEMS}) != len(ITEMS) or any(r["condition"] not in JUDGE_CONDITIONS for r in ITEMS):
    raise Stop("items.csv: item ids must be unique and every condition must be context or no_context")
QUESTIONS = {}
for r in ITEMS:
    QUESTIONS.setdefault(r["question_id"], dict(question=r["question"], context=r["context"],
                                               genre_human=CODE_OF[r["genre"]], conditions=set()))
    QUESTIONS[r["question_id"]]["conditions"].add(r["condition"])
by_genre = collections.defaultdict(list)
for qid in sorted(QUESTIONS):
    by_genre[QUESTIONS[qid]["genre_human"]].append(qid)
rng = random.Random(SEED)
PILOT_QIDS = {q for g in sorted(by_genre) for q in rng.sample(by_genre[g], min(PILOT_PER_GENRE, len(by_genre[g])))}
RUN_QIDS = PILOT_QIDS if RUN_MODE == "pilot" else set(QUESTIONS)
RUN_ITEMS = [r for r in ITEMS if r["question_id"] in RUN_QIDS]
print(f"{len(QUESTIONS)} questions, {len(ITEMS)} answers; this {RUN_MODE} run: {len(RUN_QIDS)} questions, "
      f"{len(RUN_ITEMS)} answers")
print("answers by condition:", dict(collections.Counter(r["condition"] for r in RUN_ITEMS)))
print("judge pool (locked):", ", ".join(POOL))

## The prompts (files in the dataset; golden rule 1 check: no answer text in any genre prompt)

In [ ]:
PROMPT_IDS = {"genre": "genre_v1", "judge:context": "judge_v1", "judge:no_context": "judge_noctx_v1"}
TEMPLATES = {k: (DATA_DIR / "prompts" / f"{pid}.txt").read_text(encoding="utf-8").rstrip()
             for k, pid in PROMPT_IDS.items()}
PROMPT_SHA = {k: hashlib.sha256(t.encode("utf-8")).hexdigest()[:12] for k, t in TEMPLATES.items()}
SLOT = re.compile(r"\{(context|question|answer)\}")


def fill(template, **values):
    return SLOT.sub(lambda m: values[m.group(1)], template)        # one pass: braces in a passage are safe


def genre_prompt(question, context):                               # takes no answer, by design (rule 1)
    return fill(TEMPLATES["genre"], question=question, context=context)


def judge_prompt(condition, context, question, answer):
    if condition == "context":
        return fill(TEMPLATES["judge:context"], context=context, question=question, answer=answer)
    return fill(TEMPLATES["judge:no_context"], question=question, answer=answer)


assert "{answer}" not in TEMPLATES["genre"], "the genre prompt must not have an answer slot"
assert "{context}" in TEMPLATES["judge:context"], "the with-passage judge prompt needs the passage"
assert "{context}" not in TEMPLATES["judge:no_context"], "the no-context judge prompt must not show the passage"
leaks = []
for r in ITEMS:
    q = QUESTIONS[r["question_id"]]
    a = r["answer"].strip()
    # a leak = the answer appears in the genre prompt more often than the guide text, question and passage explain
    explained = TEMPLATES["genre"].count(a) + q["question"].count(a) + q["context"].count(a)
    if a and genre_prompt(q["question"], q["context"]).count(a) > explained:
        leaks.append(r["item_id"])
if leaks:
    raise Stop(f"Golden rule 1 broken: answer text found in {len(leaks)} genre prompts, e.g. {leaks[:3]}")
print("Golden rule 1 check passed: no answer text in any genre prompt.")
print("prompt fingerprints:", PROMPT_SHA)

ex = RUN_ITEMS[0]
q0 = QUESTIONS[ex["question_id"]]
print("\n========== example: turn 1 (genre) ==========\n" + genre_prompt(q0["question"], q0["context"])[-700:])
for c in JUDGE_CONDITIONS:
    print(f"\n========== example: turn 2 (judge, {c}) ==========\n"
          + judge_prompt(c, q0["context"], q0["question"], ex["answer"]))

## The model backends (Ollama; a fake model for local tests)

In [ ]:
def sh(cmd, check=True):
    print("$", cmd)
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.stdout.strip():
        print(r.stdout.strip()[-1500:])
    if r.returncode and check:
        print(r.stderr[-3000:])
        raise Stop(f"Command failed: {cmd}")
    return r.stdout


def http_json(url, payload=None, timeout=1800):
    data = None if payload is None else json.dumps(payload).encode("utf-8")
    req = urllib.request.Request(url, data=data, headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=timeout) as resp:
        return json.loads(resp.read().decode("utf-8"))


def get_secret(name, required):
    if os.environ.get(name):
        return os.environ[name]
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        if required:
            raise Stop(f"Add the secret {name} under Add-ons -> Secrets (section 1c).")
        return None


def result(raw="", status="ok", prompt_tokens=None, output_tokens=None, done_reason="", latency=0.0,
           thinking_chars=0, error=None):
    return dict(raw=raw, status=status, prompt_tokens=prompt_tokens, output_tokens=output_tokens,
                done_reason=done_reason, latency_s=round(latency, 3), thinking_chars=thinking_chars, error=error)




class OllamaBackend:
    url = "http://127.0.0.1:11434"

    def __init__(self, cfg, parallel):
        self.cfg, self.parallel = cfg, parallel
        if not shutil.which("ollama"):
            sh("apt-get -qq update > /dev/null && apt-get -qq install -y zstd > /dev/null", check=False)
            pin = f"OLLAMA_VERSION={OLLAMA_VERSION} " if OLLAMA_VERSION else ""
            sh(f"curl -fsSL https://ollama.com/install.sh | {pin}sh")
        env = dict(os.environ, OLLAMA_NUM_PARALLEL=str(parallel), OLLAMA_MAX_LOADED_MODELS="1",
                   OLLAMA_KEEP_ALIVE="24h", OLLAMA_HOST="127.0.0.1:11434")
        self.log_path = OUT_DIR / f"ollama_server__{MODEL_KEY}.log"
        self.proc = subprocess.Popen(["ollama", "serve"], env=env, stdout=open(self.log_path, "a"),
                                     stderr=subprocess.STDOUT)
        for _ in range(90):
            try:
                self.version = http_json(self.url + "/api/version", timeout=5)["version"]
                break
            except Exception:
                time.sleep(1)
        else:
            raise Stop(f"The Ollama server did not start; read {self.log_path}")
        if OLLAMA_VERSION and self.version != OLLAMA_VERSION:
            raise Stop(f"Ollama {self.version} is running but OLLAMA_VERSION is {OLLAMA_VERSION}. Restart the session.")
        sh(f"ollama pull {cfg['model']}")
        tags = http_json(self.url + "/api/tags")["models"]
        hit = [m for m in tags if cfg["model"] in (m.get("name"), m.get("model"))]
        if not hit:
            raise Stop(f"{cfg['model']} not found after pull; installed: {[m.get('name') for m in tags]}")
        self.digest = hit[0]["digest"]
        info = http_json(self.url + "/api/show", {"model": cfg["model"]})
        self.arch = info.get("model_info", {}).get("general.architecture", "?")
        self.quant = info.get("details", {}).get("quantization_level", "?")
        if "thinking" in info.get("capabilities", []) and cfg.get("think") is None:
            cfg["think"] = False      # zero-shot like the other judges; reasoning judges set think explicitly
            print("This model can think before answering; thinking is turned OFF (think=False) for a zero-shot verdict.")
        if cfg.get("expect_arch") and self.arch != cfg["expect_arch"]:
            raise Stop(f"{cfg['model']} has architecture {self.arch}, expected {cfg['expect_arch']}. "
                       "This is not the model the paper used. Find the right GGUF before going on.")
        self.model_id = cfg["model"]
        self.version_id = f"ollama {self.version}; digest {self.digest}; {self.quant}"
        self.num_ctx = None
        print(f"Ollama {self.version} | {cfg['model']} | digest {self.digest[:16]} | arch {self.arch} | {self.quant}")

    def one(self, prompt, max_new, num_ctx):
        payload = {"model": self.cfg["model"], "messages": [{"role": "user", "content": prompt}],
                   "stream": False, "keep_alive": "24h",
                   "options": {"temperature": 0, "seed": SEED, "num_ctx": num_ctx, "num_predict": max_new}}
        if self.cfg.get("think") is not None:
            payload["think"] = self.cfg["think"]
        t, err = time.time(), None
        for attempt in range(3):
            try:
                r = http_json(self.url + "/api/chat", payload)
                break
            except Exception as e:
                err = repr(e)[:300]
                time.sleep(5 * (attempt + 1))
        else:
            return result(status="error", latency=time.time() - t, error=err)
        msg = r.get("message", {})
        pt = r.get("prompt_eval_count")
        cut = pt is not None and pt >= num_ctx - max_new - 16      # guard 1: the prompt filled the context
        return result(raw=msg.get("content", ""), status="truncated" if cut else "ok", prompt_tokens=pt,
                      output_tokens=r.get("eval_count"), done_reason=r.get("done_reason", ""),
                      latency=time.time() - t, thinking_chars=len(msg.get("thinking") or ""))

    def preflight(self, prompts):
        """Measure the longest prompts and choose num_ctx so that none is cut."""
        longest = sorted(prompts, key=len, reverse=True)[:6]
        probe_ctx = min(self.cfg["max_ctx"], 16384)
        counts = []
        for i, p in enumerate(longest):        # a unique first line stops prefix reuse from hiding tokens
            r = self.one(f"[length check {i} {time.time()}]\n" + p, 1, probe_ctx)
            if r["status"] == "error":
                raise Stop(f"The context check failed: {r['error']}")
            counts.append(r["prompt_tokens"] or 0)
        biggest, max_new = max(counts), max(self.cfg["max_new"].values())
        need = int(biggest * 1.15) + max_new + 64
        self.num_ctx = min(self.cfg["max_ctx"], int(math.ceil(need / 1024) * 1024))
        print(f"context check: longest prompt about {biggest} tokens -> num_ctx = {self.num_ctx}")
        if biggest >= probe_ctx - 8 or need > self.cfg["max_ctx"]:
            print("WARNING: the longest prompts may not fit this model. Any prompt that is cut is marked "
                  "'truncated', counted as no_answer and reported. It is never used as a verdict.")

    def generate_batch(self, prompts, max_new):
        with ThreadPoolExecutor(self.parallel) as pool:
            return list(pool.map(lambda p: self.one(p, max_new, self.num_ctx), prompts))

    def log_has_truncation(self):
        return "truncating input" in self.log_path.read_text(errors="ignore").lower()   # guard 2


def mem_report(where):
    """One line: free RAM, free disk in /tmp, free GPU memory. Shows where a memory error happens."""
    parts = [where]
    try:
        import psutil
        parts.append(f"RAM free {psutil.virtual_memory().available / 1e9:.1f} GB")
    except Exception:
        pass
    parts.append(f"disk free {shutil.disk_usage('/tmp').free / 1e9:.0f} GB")
    try:
        import torch
        for i in range(torch.cuda.device_count()):
            free, total = torch.cuda.mem_get_info(i)
            parts.append(f"GPU{i} free {free / 1e9:.1f}/{total / 1e9:.1f} GB")
    except Exception:
        pass
    print("[memory] " + " | ".join(parts))


def merge_lora_checkpoint(fetch, out_dir, r, alpha, drop=lambda path: None, part_bytes=1.5e9):
    """Some checkpoints were saved with un-merged PEFT LoRA layers (keys ...base_layer.weight,
    ...lora_A.default.weight, ...lora_B.default.weight). Plain loading then leaves every such linear layer
    uninitialized (random). This writes a normal checkpoint with W = base + (alpha / r) * B @ A.
    fetch(filename) -> local path; drop(path) frees a shard after use. Output is written in parts of about
    part_bytes, so RAM stays small (the input shard is memory-mapped). Returns out_dir."""
    import gc
    import torch
    from safetensors.torch import save_file
    out_dir = Path(out_dir)
    if (out_dir / "MERGED_OK").exists():
        return out_dir
    out_dir.mkdir(parents=True, exist_ok=True)
    index = json.loads(Path(fetch("pytorch_model.bin.index.json")).read_text(encoding="utf-8"))
    need = index.get("metadata", {}).get("total_size", 0) * 1.45
    if need and shutil.disk_usage(out_dir).free < need:
        raise Stop(f"Not enough disk to merge the LoRA layers: need about {need / 1e9:.0f} GB free in {out_dir}.")
    shards = sorted(set(index["weight_map"].values()))
    new_map, merged, parts = {}, 0, 0
    out, size = {}, 0

    def flush():
        nonlocal out, size, parts
        if out:
            parts += 1
            name = f"model-part-{parts:05d}.safetensors"
            save_file(out, str(out_dir / name), metadata={"format": "pt"})
            new_map.update({k: name for k in out})
            out, size = {}, 0
            gc.collect()

    for n, shard in enumerate(shards, 1):
        mem_report(f"merge: before shard {n}/{len(shards)}")
        path = fetch(shard)
        try:
            sd = torch.load(path, map_location="cpu", weights_only=True, mmap=True)
        except Exception as e:
            print(f"memory-mapped loading failed ({e}); loading the whole shard into RAM")
            sd = torch.load(path, map_location="cpu", weights_only=True)
        for key, t in sd.items():
            if key.endswith((".lora_A.default.weight", ".lora_B.default.weight", "rotary_emb.inv_freq")):
                continue
            if key.endswith(".base_layer.weight"):
                mod = key[: -len(".base_layer.weight")]
                a, b = sd[mod + ".lora_A.default.weight"], sd[mod + ".lora_B.default.weight"]
                if a.shape[0] != r or b.shape[1] != r:
                    raise Stop(f"{mod}: LoRA rank {a.shape[0]} is not the expected {r}, so alpha is unknown. Stop.")
                w = t.float() + (alpha / r) * (b.float() @ a.float())
                out[mod + ".weight"] = w.to(torch.float16).contiguous()
                del w
                merged += 1
            else:
                out[key] = (t.to(torch.float16) if t.is_floating_point() else t).contiguous()
            size += out[list(out)[-1]].numel() * out[list(out)[-1]].element_size()
            if size >= part_bytes:
                flush()
        flush()
        del sd
        gc.collect()
        drop(path)
    if not merged:
        raise Stop("lora_merge is set, but the checkpoint has no un-merged LoRA layers. Remove lora_merge.")
    (out_dir / "model.safetensors.index.json").write_text(json.dumps({"metadata": {}, "weight_map": new_map}))
    for f in ("config.json", "generation_config.json"):
        try:
            shutil.copyfile(fetch(f), out_dir / f)
        except Exception:
            if f == "config.json":
                raise
    (out_dir / "MERGED_OK").write_text(json.dumps(dict(r=r, alpha=alpha, merged_layers=merged)))
    print(f"merged {merged} LoRA layers (r={r}, alpha={alpha}) into {out_dir}")
    return out_dir


class HFBackend:
    def __init__(self, cfg, parallel):
        os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")   # before CUDA starts
        sh("pip install -q -U bitsandbytes accelerate", check=False)
        import torch
        from huggingface_hub import HfApi
        from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
        token = get_secret("HF_TOKEN", required=False)
        self.cfg, self.batch, self.torch = cfg, parallel, torch
        self.sha = HfApi().model_info(cfg["model"], revision=cfg["revision"], token=token).sha
        self.tok = AutoTokenizer.from_pretrained(cfg["model"], revision=self.sha, token=token)
        self.tok.padding_side = "left"
        if self.tok.pad_token is None:
            self.tok.pad_token = self.tok.eos_token
        quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                   bnb_4bit_compute_dtype=torch.float16)
        source, where, merge_note = cfg["model"], dict(revision=self.sha, token=token), ""
        if cfg.get("lora_merge"):
            os.environ.setdefault("HF_XET_CHUNK_CACHE_SIZE_BYTES", "0")      # no second copy of 27 GB on disk
            from huggingface_hub import hf_hub_download
            dl = Path("/tmp/hf_download")
            mem_report("before merge")
            source = merge_lora_checkpoint(
                lambda f: hf_hub_download(cfg["model"], f, revision=self.sha, token=token, local_dir=str(dl)),
                Path("/tmp") / f"merged_{self.sha[:12]}", drop=lambda p: Path(p).unlink(missing_ok=True),
                **cfg["lora_merge"])
            where, merge_note = {}, f"; LoRA merged r={cfg['lora_merge']['r']} alpha={cfg['lora_merge']['alpha']}"
        mem_report("before loading the model in 4-bit")
        self.model, info = AutoModelForCausalLM.from_pretrained(source, **where, quantization_config=quant,
                                                                device_map="auto", torch_dtype=torch.float16,
                                                                output_loading_info=True)
        self.model.eval()
        mem_report("model loaded")
        from transformers.utils import logging as hf_logging
        hf_logging.set_verbosity_error()        # silence the per-batch "max_new_tokens and max_length" notice
        # memory for one batch grows with (prompts x tokens); LLaMA-2-13B keeps ~0.8 MB of cache per token
        self.batch_tokens = cfg.get("batch_tokens", 8192)
        missing = [k for k in info.get("missing_keys", []) if not k.endswith("rotary_emb.inv_freq")]
        if missing:                                     # never judge with uninitialized (random) weights
            raise Stop(f"{len(missing)} weights were not in the checkpoint, e.g. {missing[:3]}. They would be random. "
                       "If the names end in base_layer/lora_A/lora_B, set lora_merge for this model.")
        self.fmt = "chat_template" if self.tok.chat_template else "alpaca"
        self.model_id, self.quant, self.num_ctx = cfg["model"], "bnb-nf4", cfg["max_ctx"]
        self.version_id = f"hf revision {self.sha}; {self.quant}{merge_note}; format {self.fmt}"
        print(f"{cfg['model']} @ {self.sha[:12]} | prompt format: {self.fmt}")

    def render(self, prompt):
        if self.fmt == "chat_template":
            return self.tok.apply_chat_template([{"role": "user", "content": prompt}], tokenize=False,
                                                add_generation_prompt=True)
        return f"### Instruction:\n{prompt}\n\n### Response:\n"

    def ids(self, prompt):
        return self.tok(self.render(prompt), add_special_tokens=self.fmt == "alpaca")["input_ids"]

    def preflight(self, prompts):
        lens = [len(self.ids(p)) for p in prompts]
        over = sum(n + max(self.cfg["max_new"].values()) > self.num_ctx for n in lens)
        print(f"context check: longest prompt {max(lens)} tokens; {over} of {len(lens)} prompts do not fit "
              f"{self.num_ctx} tokens and will be marked too_long (never cut)")

    def generate_batch(self, prompts, max_new):
        out, enc = [None] * len(prompts), [self.ids(p) for p in prompts]
        ok = []
        for i, e in enumerate(enc):
            if len(e) + max_new > self.num_ctx:
                out[i] = result(status="too_long", prompt_tokens=len(e))
            else:
                ok.append(i)
        ok.sort(key=lambda i: len(enc[i]))
        groups, cur = [], []
        for i in ok:                    # sorted by length: the newest prompt is the longest in its group
            if cur and (len(cur) >= self.batch or (len(cur) + 1) * (len(enc[i]) + max_new) > self.batch_tokens):
                groups.append(cur)
                cur = []
            cur.append(i)
        if cur:
            groups.append(cur)
        for idx in groups:
            self.run_group(idx, enc, out, max_new)
        return out

    def run_group(self, idx, enc, out, max_new):
        """Generate for one group; if the GPU runs out of memory, split the group in two and try again."""
        batch = self.tok.pad({"input_ids": [enc[i] for i in idx]}, return_tensors="pt").to(self.model.device)
        t = time.time()
        try:
            with self.torch.no_grad():
                gen = self.model.generate(**batch, max_new_tokens=max_new, do_sample=False,
                                          pad_token_id=self.tok.pad_token_id)
        except self.torch.cuda.OutOfMemoryError:
            del batch
            self.torch.cuda.empty_cache()
            if len(idx) == 1:
                out[idx[0]] = result(status="error", prompt_tokens=len(enc[idx[0]]),
                                     error="CUDA out of memory even for a single prompt")
                return
            print(f"GPU memory full for {len(idx)} prompts of up to {max(len(enc[i]) for i in idx)} tokens; "
                  "splitting the batch")
            half = len(idx) // 2
            self.run_group(idx[:half], enc, out, max_new)
            self.run_group(idx[half:], enc, out, max_new)
            return
        dt = (time.time() - t) / len(idx)
        for j, i in enumerate(idx):
            new = gen[j, batch["input_ids"].shape[1]:]
            n_out = int((new != self.tok.pad_token_id).sum())
            out[i] = result(raw=self.tok.decode(new, skip_special_tokens=True), prompt_tokens=len(enc[i]),
                            output_tokens=n_out, done_reason="length" if n_out >= max_new else "stop", latency=dt)


class OpenAIBackend:
    def __init__(self, cfg, parallel):
        sh("pip install -q openai", check=False)
        from openai import OpenAI
        self.client = OpenAI(api_key=get_secret("OPENAI_API_KEY", required=True))
        self.cfg, self.parallel = cfg, parallel
        self.model_id, self.quant, self.num_ctx = cfg["model"], "api", cfg["max_ctx"]
        self.version_id = f"openai {cfg['model']}"

    def preflight(self, prompts):
        print("context check: not needed (the API context is far larger than any prompt)")

    def one(self, prompt, max_new):
        t, err = time.time(), None
        for attempt in range(6):
            try:
                r = self.client.chat.completions.create(model=self.cfg["model"], temperature=0, seed=SEED,
                                                        max_tokens=max_new,
                                                        messages=[{"role": "user", "content": prompt}])
                c = r.choices[0]
                return result(raw=c.message.content or "", prompt_tokens=r.usage.prompt_tokens,
                              output_tokens=r.usage.completion_tokens, done_reason=c.finish_reason,
                              latency=time.time() - t)
            except Exception as e:
                err = repr(e)[:300]
                time.sleep(min(60, 2 ** attempt))
        return result(status="error", latency=time.time() - t, error=err)

    def generate_batch(self, prompts, max_new):
        with ThreadPoolExecutor(self.parallel) as pool:
            return list(pool.map(lambda p: self.one(p, max_new), prompts))


class MockBackend:
    """Local tests only: a fake model with a few badly formatted replies, to exercise parsing."""

    def __init__(self, cfg, parallel):
        self.model_id, self.quant, self.num_ctx, self.version_id = "mock", "none", 32768, "mock 1"

    def preflight(self, prompts):
        print("context check: mock")

    def one(self, prompt, max_new):
        h = int(hashlib.md5((MODEL_KEY + prompt).encode("utf-8")).hexdigest(), 16)
        if prompt.rstrip().endswith("Genre:"):
            raw = ["HISTORY", "ARTS", "SCIENCE", "GEOGRAPHY", "RELIGION", "OTHER", "History.", "I think ARTS"][h % 8]
        elif "Context: " in prompt:
            ctx = prompt.split("Context: ", 1)[1].split("\nQuestion: ", 1)[0]
            ans = prompt.split("Model answer: ", 1)[1].rsplit("\nAnswer now:", 1)[0]
            raw = "no" if ans.strip() in ctx else "yes"
            raw = {0: "Yes.", 1: "The answer is no"}.get(h % 25, raw)
        else:                                       # no passage: the fake model guesses
            raw = {0: "Yes.", 1: "I am not sure"}.get(h % 25, ["yes", "no"][h % 2])
        return result(raw=raw, prompt_tokens=len(prompt) // 3, output_tokens=2, done_reason="stop", latency=0.001)

    def generate_batch(self, prompts, max_new):
        return [self.one(p, max_new) for p in prompts]


BACKENDS = {"ollama": OllamaBackend, "hf": HFBackend, "openai": OpenAIBackend, "mock": MockBackend}
backend = None        # started for each model by use_model() below

## Logging, parsing and resuming (one log line per call; strict parse; `no_answer` counted separately)

In [ ]:
THINK_BLOCK = re.compile(r"<think>.*?</think>", re.S)
EDGE = " \t\r\n\"'`*.,!:;"


def strip_think(text):
    text = THINK_BLOCK.sub("", text or "")
    return "" if "<think>" in text else text        # an unfinished thinking block has no final answer


def parse(stage, raw, strict=True):
    text = strip_think(raw)
    if strict:
        t = text.strip(EDGE)
        if stage == "judge":
            return t.lower() if t.lower() in ("yes", "no") else "no_answer"
        if t.upper() in GENRES:
            return t.upper()
        return CODE_OF.get(next((n for n in GENRES.values() if n.lower() == t.lower()), ""), "no_answer")
    words = re.findall(r"[A-Za-z]+", text)
    keys = ("yes", "no") if stage == "judge" else tuple(GENRES)
    for w in words:
        w2 = w.lower() if stage == "judge" else w.upper()
        if w2 in keys:
            return w2
    return "no_answer"


def base(key):
    return key.split(":")[0]                       # "judge:no_context" -> "judge"


def setup_fingerprint(key):
    parts = dict(dataset=DATASET, model_key=MODEL_KEY, model=CFG["model"], backend=CFG["backend"],
                 version=backend.version_id, prompt=PROMPT_SHA[key], temperature=0, seed=SEED,
                 think=CFG.get("think"), max_new=CFG["max_new"][base(key)])
    return hashlib.sha256(json.dumps(parts, sort_keys=True).encode()).hexdigest()[:16]


def prompt_hash(prompt):
    return hashlib.sha256(prompt.encode("utf-8")).hexdigest()[:16]


def run_path(key):
    return RUN_DIR / f"{DATASET}__{MODEL_KEY}__{base(key)}__{PROMPT_IDS[key]}.jsonl"


def import_previous_runs():
    """Merge run files from earlier runs that were added with 'Add Input': a notebook's output, or a dataset made
    from an outputs zip. Searched at any folder depth; a file seen twice adds nothing."""
    root = os.environ.get("BENCH_INPUT_DIR", "/kaggle/input")
    found = [f for f in glob.glob(f"{root}/**/runs/{DATASET}__{MODEL_KEY}__*.jsonl", recursive=True)
             if any(f.endswith(f"__{pid}.jsonl") for pid in PROMPT_IDS.values())]   # only this notebook's prompts
    if not found:
        print(f"no earlier runs of {MODEL_KEY} among the inputs (a fresh start)")
    for src in sorted(set(found)):
        dst = RUN_DIR / Path(src).name
        have = set()
        if dst.exists():
            have = {json.loads(l)["key"] + json.loads(l)["status"] for l in open(dst, encoding="utf-8") if l.strip()}
        new = [l for l in open(src, encoding="utf-8")
               if l.strip() and json.loads(l)["key"] + json.loads(l)["status"] not in have]
        with open(dst, "a", encoding="utf-8") as f:
            f.writelines(new)
        print(f"resumed {len(new)} records from {src}")


def load_records(stage):
    recs = {}
    if run_path(stage).exists():
        for line in open(run_path(stage), encoding="utf-8"):
            if line.strip():
                r = json.loads(line)
                recs[r["key"]] = r                     # a later line (a retry) replaces an earlier one
    return recs


SESSION = {}


def run_stage(stage, jobs):
    """stage: "genre", "judge:context" or "judge:no_context". jobs: list of (key, question_id, item_id, prompt).
    Returns True if every job is done."""
    fp, max_new = setup_fingerprint(stage), CFG["max_new"][base(stage)]
    done = load_records(stage)
    other = {r["setup_hash"] for r in done.values()} - {fp}
    if other:
        raise Stop(f"{run_path(stage).name} has records from a different setup {other} (this setup: {fp}). "
                   "Never mix setups: use the same OLLAMA_VERSION/model/prompt, or start a new study folder.")
    todo = [j for j in jobs if j[0] not in done or done[j[0]]["status"] == "error"
            or done[j[0]].get("prompt_hash") != prompt_hash(j[3])]
    print(f"[{stage}] {len(jobs)} calls in this {RUN_MODE} run: {len(jobs) - len(todo)} already done, {len(todo)} to do")
    t_start, n_done, chunk = time.time(), 0, max(16, PAR * 8)
    with open(run_path(stage), "a", encoding="utf-8") as f:
        for s in range(0, len(todo), chunk):
            if time.time() - T0 > TIME_BUDGET_HOURS * 3600:
                print(f"[{stage}] TIME BUDGET reached: stopping cleanly. Resume in a new version (section 2, step 5).")
                SESSION[stage] = dict(calls=n_done, seconds=time.time() - t_start)
                return False
            batch = todo[s:s + chunk]
            outs = backend.generate_batch([j[3] for j in batch], max_new)
            for (key, qid, item_id, prompt), o in zip(batch, outs):
                capped = bool(CFG.get("think")) and o["done_reason"] == "length"
                usable = o["status"] == "ok" and not capped
                rec = dict(dataset=DATASET, stage=base(stage), condition=stage.split(":")[1] if ":" in stage else None,
                           key=key, question_id=qid, item_id=item_id,
                           model_key=MODEL_KEY, model_id=backend.model_id, model_version=backend.version_id,
                           backend=CFG["backend"], prompt_id=PROMPT_IDS[stage], prompt_sha=PROMPT_SHA[stage],
                           temperature=0, seed=SEED, think=CFG.get("think"), num_ctx=backend.num_ctx,
                           max_new_tokens=max_new, prompt_hash=prompt_hash(prompt), timestamp=datetime.datetime.now(datetime.timezone.utc).isoformat(),
                           raw_output=o["raw"],
                           parsed=parse(base(stage), o["raw"]) if usable else "no_answer",
                           parsed_lenient=parse(base(stage), o["raw"], strict=False) if usable else "no_answer",
                           status="capped" if capped else o["status"], prompt_tokens=o["prompt_tokens"],
                           output_tokens=o["output_tokens"], thinking_chars=o["thinking_chars"],
                           done_reason=o["done_reason"], latency_s=o["latency_s"], error=o["error"],
                           setup_hash=fp, run_mode=RUN_MODE)
                f.write(json.dumps(rec, ensure_ascii=False) + "\n")
            f.flush()
            n_done += len(batch)
            errors = sum(o["status"] == "error" for o in outs)
            if errors > 0.2 * len(batch):
                raise Stop(f"[{stage}] {errors} of {len(batch)} calls failed, e.g. {outs[0]['error']}. "
                           "Check the server log or the API key.")
            if n_done % (chunk * 10) < chunk or n_done == len(todo):
                rate = n_done / max(1e-9, time.time() - t_start)
                print(f"[{stage}] {n_done}/{len(todo)} | {rate:.2f} calls/s | about "
                      f"{(len(todo) - n_done) / max(rate, 1e-9) / 60:.0f} min left")
    SESSION[stage] = dict(calls=n_done, seconds=time.time() - t_start)
    return True

## Routes and model switching

- The prompts must be exactly the ones every BenHalluEval run used (the lock stores their fingerprints).
- `use_model` stops the old Ollama server, starts one with the next model's settings, downloads the model and measures
  the longest prompt it will see, so no prompt is cut. Models are deleted when the disk gets full.
- The routes follow the lock's rules exactly. A genre without a usable cell falls back to A; ties go to the better
  overall judge.

In [ ]:
for key, lock_key in (("genre", "genre"), ("judge:context", "context"), ("judge:no_context", "no_context")):
    if PROMPT_SHA[key] != LOCK["setup"]["prompt_sha12"][lock_key]:
        raise Stop(f"prompt {PROMPT_IDS[key]} differs from the one in the lock: use the exact BenHalluEval prompts")

ALL_D2_PROMPTS = ([genre_prompt(QUESTIONS[q]["question"], QUESTIONS[q]["context"]) for q in sorted(RUN_QIDS)]
                  + [judge_prompt(r["condition"], r["context"], r["question"], r["answer"]) for r in RUN_ITEMS])
COST, VERSIONS, LOADED, DONE = [], {}, None, {}


def free_disk_gb():
    return shutil.disk_usage(Path.home()).free / 1e9


def installed_models():
    try:
        return [m["name"] for m in http_json(OllamaBackend.url + "/api/tags", timeout=10)["models"]]
    except Exception:
        return []


def make_room(keep, need_gb=30):
    """Delete downloaded models (those not in `keep` first) until `need_gb` GB are free."""
    if not isinstance(backend, OllamaBackend):
        return
    keep_names = {MODELS[k]["model"] for k in keep}
    for name in sorted(installed_models(), key=lambda n: n in keep_names):
        if free_disk_gb() >= need_gb:
            break
        sh(f"ollama rm {name}", check=False)
        print(f"deleted {name} to free disk ({free_disk_gb():.0f} GB free)")


def use_model(key, keep=()):
    global MODEL_KEY, CFG, PAR, backend, LOADED
    if LOADED == key:
        return
    make_room(keep)
    if getattr(backend, "proc", None):
        backend.proc.terminate()
        backend.proc.wait(timeout=120)
    MODEL_KEY = key
    CFG = {**DEFAULTS, **MODELS[key]}
    if "claude" in CFG["model"].lower():
        raise Stop("Claude wrote the hallucinated answers: never use it as a judge or guesser.")
    if os.environ.get("BENCH_BACKEND_OVERRIDE"):          # local tests only
        CFG["backend"] = os.environ["BENCH_BACKEND_OVERRIDE"]
    PAR = PARALLEL or CFG["parallel"]
    print(f"\n==================== {key} ({CFG['model']}) ====================")
    backend = BACKENDS[CFG["backend"]](CFG, PAR)
    backend.preflight(ALL_D2_PROMPTS)
    VERSIONS[key] = dict(model_id=backend.model_id, version=backend.version_id, num_ctx=backend.num_ctx, parallel=PAR)
    LOADED = key
    import_previous_runs()


def run(stage, jobs):
    if not jobs:
        return True
    if time.time() - T0 > TIME_BUDGET_HOURS * 3600:
        print(f"[{MODEL_KEY} {stage}] TIME BUDGET reached before this turn. Resume in a new version.")
        DONE[(MODEL_KEY, stage)] = False
        return False
    DONE[(MODEL_KEY, stage)] = run_stage(stage, jobs)
    s = SESSION.get(stage, {})
    COST.append(dict(model=MODEL_KEY, turn=stage, jobs=len(jobs), calls=s.get("calls", 0),
                     seconds=round(s.get("seconds", 0), 1)))
    return DONE[(MODEL_KEY, stage)]


def records(key, stage, jobs):
    """This model's finished records for exactly these prompts (a record for another prompt text is stale)."""
    want = {j[0]: prompt_hash(j[3]) for j in jobs}
    p = RUN_DIR / f"{DATASET}__{key}__{base(stage)}__{PROMPT_IDS[stage]}.jsonl"
    out = {}
    if p.exists():
        for line in open(p, encoding="utf-8"):
            if line.strip():
                r = json.loads(line)
                if want.get(r["key"]) == r.get("prompt_hash") and r["status"] != "error":
                    out[r["key"]] = r
    return out


def judge_jobs(rows, condition):
    return [(r["item_id"], r["question_id"], r["item_id"], judge_prompt(condition, r["context"], r["question"], r["answer"]))
            for r in rows if r["condition"] == condition]


L, RB = LOCK["conditions"], LOCK["R_best_path"]["per_condition"]
TABLE = {c: {g: v["judge"] for g, v in L[c]["D_E_judge_per_genre"].items() if v.get("BHS") is not None} for c in L}
TSCORE = {c: {g: v["BHS"] for g, v in L[c]["D_E_judge_per_genre"].items() if v.get("BHS") is not None} for c in L}
OWN = {c: {(j, g): v["BHS"] for j, cells in L[c]["F_own_genre_scores"].items() for g, v in cells.items()} for c in L}
OVERALL = {c: {j: v["BHS"] for j, v in L[c]["overall_BHS"].items()} for c in L}
A_OF = {c: L[c]["A_best_single_judge"] for c in L}
GUESSER = {c: L[c]["best_genre_guesser"]["judge"] for c in L}
ASKERS = {c: RB[c]["askers"] for c in L}


def route(method, condition, qid, named):
    """named: {LLM: genre code it named for this question} (askers only)."""
    A = A_OF[condition]
    if method == "A":
        return A
    if method == "D":
        return TABLE[condition].get(QUESTIONS[qid]["genre_human"], A)
    g = named.get(GUESSER[condition])
    if method == "E":
        return TABLE[condition].get(g, A)
    own = [(OWN[condition][(j, named.get(j))], OVERALL[condition][j], j) for j in ASKERS[condition]
           if (j, named.get(j)) in OWN[condition]]
    if method == "F":
        return min(own)[2] if own else A
    if method == "R":
        ours = ((TSCORE[condition][g], OVERALL[condition][TABLE[condition][g]], TABLE[condition][g]) if g in TABLE[condition]
                else (OVERALL[condition][A], OVERALL[condition][A], A))
        return min(own + [ours])[2]
    raise ValueError(method)


# who names the genre of which question: the askers of each condition the question is judged in
ASK_Q = collections.defaultdict(list)
if any(m in METHODS for m in "REF"):
    for q in sorted(RUN_QIDS):
        for c in sorted(QUESTIONS[q]["conditions"]):
            for k in ASKERS[c]:
                if q not in ASK_Q[k]:
                    ASK_Q[k].append(q)
GENRE_JOBS = {k: [(q, q, None, genre_prompt(QUESTIONS[q]["question"], QUESTIONS[q]["context"])) for q in qs]
              for k, qs in ASK_Q.items()}
print("prompts match the lock:", {k: PROMPT_SHA[k] for k in PROMPT_IDS})
print("A:", A_OF, "| best genre guesser:", GUESSER)
print("LLMs asked for the genre:", {k: len(v) for k, v in ASK_Q.items()} or "none (route A needs no genre)")

In [ ]:
NAMED, GENRE_COMPLETE = {}, True     # route A needs no genre

## Turn 2: the single best judge judges every answer

Each answer is judged once per picked judge, in its own condition. The routes are saved in `routes.csv`.

In [ ]:
ROUTES, NEED = [], collections.defaultdict(list)
if GENRE_COMPLETE:
    for r in RUN_ITEMS:
        named = {k: NAMED[k].get(r["question_id"]) for k in NAMED}
        row = dict(item_id=r["item_id"], question_id=r["question_id"], condition=r["condition"],
                   our_genre=QUESTIONS[r["question_id"]]["genre_human"])
        for m in METHODS:
            row[m] = route(m, r["condition"], r["question_id"], named)
            if r not in NEED[row[m]]:
                NEED[row[m]].append(r)
        row.update({f"named_{k}": named[k] for k in sorted(NAMED) if k in ASKERS[r["condition"]]})
        ROUTES.append(row)
    order = sorted(NEED, key=lambda k: (k != LOADED, -len(NEED[k])))
    for c in JUDGE_CONDITIONS:
        for m in METHODS:
            print(f"route {m}, {c}: " + ", ".join(f"{k} {n}" for k, n in collections.Counter(
                row[m] for row in ROUTES if row["condition"] == c).most_common()))
    for k in order:
        if time.time() - T0 > TIME_BUDGET_HOURS * 3600:
            print("TIME BUDGET reached in turn 2. Resume in a new version.")
            break
        use_model(k, keep=order[order.index(k):])
        for c in JUDGE_CONDITIONS:
            run(f"judge:{c}", judge_jobs(NEED[k], c))

VERDICT = {}
for k, rows in NEED.items():
    for c in JUDGE_CONDITIONS:
        for item_id, rec in records(k, f"judge:{c}", judge_jobs(rows, c)).items():
            VERDICT[(k, item_id)] = rec
PARTIAL = (not GENRE_COMPLETE or any(v is False for v in DONE.values())
           or any((row[m], row["item_id"]) not in VERDICT for row in ROUTES for m in METHODS))

## Summary, cost and the outputs zip

The preview below scores against the **design labels** (gold = correct, written hallucination = hallucinated). The
final scores use the annotators' agreed labels and are computed by `src/dataset2/compare_routes.py`.

In [ ]:
def md_table(rows, cols):
    return "| " + " | ".join(cols) + " |\n|" + "---|" * len(cols) + "\n" + "\n".join(
        "| " + " | ".join(str(r.get(c, "")) for c in cols) + " |" for r in rows)


def preview_bhs(rows, method):
    s = [0, 0, 0, 0]
    for row in rows:
        v = VERDICT.get((row[method], row["item_id"]), {}).get("parsed", "no_answer")
        if ITEM_LABEL[row["item_id"]] == 0:
            s[0] += 1; s[1] += v != "no"
        else:
            s[2] += 1; s[3] += v != "yes"
    return round(50 * (s[1] / s[0] + s[3] / s[2]), 2) if s[0] and s[2] else None


ITEM_LABEL = {r["item_id"]: int(r["design_label"]) for r in RUN_ITEMS}
if ROUTES:
    prev = []
    for c in JUDGE_CONDITIONS + ["both"]:
        rows = [row for row in ROUTES if c == "both" or row["condition"] == c]
        for m in METHODS:
            na = sum(VERDICT.get((row[m], row["item_id"]), {}).get("parsed", "no_answer") == "no_answer" for row in rows)
            prev.append(dict(condition=c, route=m, answers=len(rows), questions=len({row["question_id"] for row in rows}),
                             no_answer=na, BHS_preview_design_labels=preview_bhs(rows, m)))
    show_md("**Preview (design labels, not the final result)**\n\n" + md_table(
        prev, ["condition", "route", "answers", "questions", "no_answer", "BHS_preview_design_labels"]))
cut = sum(rec["status"] in ("truncated", "too_long") for rec in VERDICT.values())
logs_cut = [p.name for p in OUT_DIR.glob("ollama_server__*.log") if "truncating input" in p.read_text(errors="ignore").lower()]
if cut or logs_cut:
    show_md(f"**WARNING: {cut} prompts were cut; server logs with truncation: {logs_cut}.** They count as no_answer.")
show_md("**Cost (this session)**\n\n" + md_table(COST, ["model", "turn", "jobs", "calls", "seconds"]))

out = RES_DIR / NOTEBOOK / RUN_MODE
out.mkdir(parents=True, exist_ok=True)
if ROUTES:
    cols = list(dict.fromkeys(k for row in ROUTES for k in row))
    with open(out / "routes.csv", "w", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=cols)
        w.writeheader()
        w.writerows(ROUTES)
manifest = dict(notebook=NOTEBOOK, methods=METHODS, run_mode=RUN_MODE, models=VERSIONS, pool=POOL,
                askers={k: len(v) for k, v in ASK_Q.items()}, prompt_ids=PROMPT_IDS, prompt_sha=PROMPT_SHA,
                lock_sha256=LOCK_SHA256, seed=SEED, temperature=0, dataset_dir=str(DATA_DIR), dataset_checksums=sums,
                questions=len(RUN_QIDS), answers=len(RUN_ITEMS),
                started=datetime.datetime.fromtimestamp(T0, datetime.timezone.utc).isoformat(),
                finished=datetime.datetime.now(datetime.timezone.utc).isoformat(), partial=PARTIAL,
                truncated=cut, cost=COST)
(out / "manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=1, default=str), encoding="utf-8")
zpath = OUT_DIR / f"{DATASET}__{NOTEBOOK}__{RUN_MODE}__outputs.zip"
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(RUN_DIR.glob(f"{DATASET}__*.jsonl")) + sorted(out.glob("*")) + sorted(OUT_DIR.glob("ollama_server__*.log")):
        z.write(p, p.relative_to(OUT_DIR))
print("download:", zpath)
show_md(("## PARTIAL: resume before using these numbers (step 5 at the top)." if PARTIAL else "## Done.")
        + f"\n\nNotebook `{NOTEBOOK}`, mode `{RUN_MODE}`, judges used: {', '.join(VERSIONS)}.")